# ChnSentiCorp_htl_all 資料集探索與平衡語料構造

## 學習目標

完成本 notebook 後，你將能夠：

1. 說明 ChnSentiCorp_htl_all 資料集的欄位結構與類別分布
2. 使用 `datasets` 函式庫的 `load_dataset` 從 Hugging Face Hub 載入中文情感分析資料集，理解為何它優於 pandas 直讀 CSV
3. 使用 `train_test_split(stratify_by_column, seed=42)` 構造類別平衡的子語料，理解 stratify 的必要性
4. 使用 `evaluate.load` 計算類別分布統計，養成可重現的實驗習慣

## 前置知識

- Python 基礎（list comprehension、dict）
- 對分類任務與 label 概念有初步了解

## 相鄰 notebook 銜接

| 方向 | 路徑 | 說明 |
|------|------|------|
| 上游 | `../03.Model.ipynb` | 介紹 AutoModel / AutoTokenizer 基本用法 |
| 下游 | `../../04Datasets/04 Datasets.ipynb` | 以本資料集示範 `datasets` 完整 map/filter/tokenize 流程 |
| 延伸 | `../../06Trainer/06 Trainer classification_demo.ipynb` | 以本資料集訓練情感分類器 |

---

## 資料集說明

| 項目 | 內容 |
|------|------|
| 名稱 | ChnSentiCorp_htl_all |
| 來源 | 攜程網（ctrip.com），由 [譚松波](http://people.ucas.ac.cn/~0012244) 老師整理 |
| 規模 | 約 7,000 筆酒店評論（正向 ~5,000、負向 ~2,000） |
| 任務 | 情感/觀點傾向性分析（二元分類） |
| 加工 | 合併萬個離散檔為單一 CSV、負向 label 從 -1 改為 0、去重 |
| HF Hub | `liyucheng/ChnSentiCorp_htl_all` |

### 欄位說明

| 欄位 | 型別 | 說明 |
|------|------|------|
| `label` | int (0/1) | 1 = 正向評論，0 = 負向評論 |
| `review` | str | 評論本文 |

In [ ]:
# --- 版本鎖定 (2026 統一慣例) ---
# 確保整個 repo 使用相同的函式庫版本，可重現實驗結果。
# 安裝後如有 kernel 重啟提示，請重啟再繼續執行後續 cells。
%pip install -q \
    "datasets>=3.0" \
    "evaluate>=0.4" \
    "pandas>=2.0" \
    "scikit-learn>=1.4"

## 1. 環境設定與可重現性

### WHY：為什麼要 `set_seed` 與統一匯入？

- `set_seed(42)` 讓 `datasets` 內部的隨機抽樣（shuffle、sample）與 Python/NumPy 亂數保持一致，確保每次執行結果相同。
- 在資料探索階段就鎖定亂數，能讓後續 notebook（訓練、評測）在相同的 split 下比較結果，避免「每次跑結果不一樣」的困惑。

In [ ]:
from datasets import load_dataset, ClassLabel
from transformers import set_seed
import pandas as pd

SEED = 42
set_seed(SEED)

## 2. 從 Hugging Face Hub 載入資料集

### WHY：使用 `load_dataset` 的原因

```python
ds = load_dataset("liyucheng/ChnSentiCorp_htl_all")
```

- 從 HF Hub 下載並自動快取，任何機器都可重現，無需管理本機路徑
- 回傳 `DatasetDict`，可直接接 `.map()`、`.filter()`、Trainer 的 `train_dataset` 參數
- 支援 `datasets 3.x` 的 Arrow 記憶體映射格式，百萬筆資料也不會 OOM
- `batched=True` 的 map 底層使用 Apache Arrow 向量化，效能遠優於逐列迭代

> **注意**：若 HF Hub 無法連線，可改用本機 CSV：
> ```python
> from pathlib import Path
> import os
> data_path = Path(os.environ.get("DATA_DIR", "./data")) / "ChnSentiCorp_htl_all.csv"
> ds = load_dataset("csv", data_files=str(data_path))
> ```

In [ ]:
DATASET_ID = "liyucheng/ChnSentiCorp_htl_all"

# load_dataset returns a DatasetDict; "train" is the only split in this dataset.
ds = load_dataset(DATASET_ID)
print(ds)

## 3. 資料概覽

先確認欄位型別與類別分布，這是每次使用新資料集的第一步。
類別不平衡（正向 ~5000、負向 ~2000）是後續需要處理的核心問題。

In [ ]:
# Inspect the dataset schema (column types, number of rows)
print("Schema:", ds["train"].features)
print(f"Total rows: {len(ds['train'])}")

# Class distribution
labels = ds["train"]["label"]
n_pos = sum(1 for l in labels if l == 1)
n_neg = sum(1 for l in labels if l == 0)
print(f"Positive (label=1): {n_pos}")
print(f"Negative (label=0): {n_neg}")
print(f"Imbalance ratio (pos/neg): {n_pos / n_neg:.2f}")

In [ ]:
# Quick peek at 20 random samples — use seed for reproducibility
sample = ds["train"].shuffle(seed=SEED).select(range(20))
df_sample = sample.to_pandas()
df_sample

## 4. 構造類別平衡語料

### WHY：為什麼需要平衡語料？為什麼用 `train_test_split(stratify_by_column)`？

原始資料集正/負比約 2.5:1。若直接切 train/validation，可能在某個 fold 裡負向樣本比例更低，導致：
- 模型偏向預測多數類（正向），F1 score 虛高
- Validation loss 無法真實反映模型對少數類的學習品質

2026 的做法是使用 `datasets` 原生 API 構造平衡語料：

```python
balanced = ds["train"].train_test_split(
    test_size=0.5,
    stratify_by_column="label",
    seed=42
)
```

`stratify_by_column` 確保每個 split 的 label 比例與原始分布一致。這是分類任務 data split 的正確預設行為，`seed=42` 鎖定亂數確保可重現。

### 構造三種規模的平衡語料

下面構造 ba_2000 / ba_4000 / ba_6000 三種規模，模擬「低資源」到「中資源」場景的學習曲線比較。

In [ ]:
def make_balanced_split(dataset, total_size: int, seed: int = SEED):
    """
    Construct a balanced corpus of `total_size` samples (50% positive, 50% negative).

    If one class has fewer samples than total_size // 2, sampling is done with
    replacement to avoid an empty dataset.

    Args:
        dataset: A Hugging Face Dataset with a 'label' column (0/1).
        total_size: Total number of samples in the balanced corpus.
        seed: Random seed for reproducibility.

    Returns:
        A Dataset with equal positive and negative samples.
    """
    half = total_size // 2

    pos = dataset.filter(lambda x: x["label"] == 1)
    neg = dataset.filter(lambda x: x["label"] == 0)

    # select with replacement when the class is smaller than required
    def sample_class(subset, n, s):
        if len(subset) >= n:
            return subset.shuffle(seed=s).select(range(n))
        # with replacement: repeat indices
        import random
        rng = random.Random(s)
        indices = [rng.randrange(len(subset)) for _ in range(n)]
        return subset.select(indices)

    pos_sample = sample_class(pos, half, seed)
    neg_sample = sample_class(neg, half, seed + 1)  # different seed to avoid overlap

    from datasets import concatenate_datasets
    balanced = concatenate_datasets([pos_sample, neg_sample]).shuffle(seed=seed)

    n_pos = sum(1 for l in balanced["label"] if l == 1)
    n_neg = sum(1 for l in balanced["label"] if l == 0)
    print(f"Balanced corpus (total={len(balanced)}): positive={n_pos}, negative={n_neg}")
    return balanced

In [ ]:
# ba_2000: 1000 positive + 1000 negative
ba_2000 = make_balanced_split(ds["train"], total_size=2000)
ba_2000.to_pandas().sample(10, random_state=SEED)

In [ ]:
# ba_4000: 2000 positive + 2000 negative
ba_4000 = make_balanced_split(ds["train"], total_size=4000)
ba_4000.to_pandas().sample(10, random_state=SEED)

In [ ]:
# ba_6000: 3000 positive + 3000 negative
ba_6000 = make_balanced_split(ds["train"], total_size=6000)
ba_6000.to_pandas().sample(10, random_state=SEED)

## 5. 使用 `train_test_split` 構造可直接訓練的 DatasetDict

### WHY：為什麼在資料探索階段就做 split？

許多初學者在訓練前才做 split，但正確的做法是：
**資料探索完成後立即 split，之後只看 train split，validation/test 保持「未見」狀態。**

否則你可能在探索中發現 test 資料的分布特徵並不自覺地偏向它，這就是**資料洩漏（data leakage）**。

`stratify_by_column="label"` 確保 train/test 各自保持與原始資料集相同的正/負比例，避免因隨機切割造成比例失衡。

In [ ]:
# Split ba_6000 into train/validation for downstream training notebooks.
# stratify_by_column ensures label distribution is preserved in each split.
ba_6000_split = ba_6000.train_test_split(
    test_size=0.15,
    stratify_by_column="label",
    seed=SEED,
)
print(ba_6000_split)

# Verify label balance in each split
for split_name, split_ds in ba_6000_split.items():
    labels = split_ds["label"]
    n_pos = sum(1 for l in labels if l == 1)
    n_neg = sum(1 for l in labels if l == 0)
    print(f"{split_name}: total={len(split_ds)}, positive={n_pos}, negative={n_neg}")

## 小結

本 notebook 示範了 2026 年的資料集探索慣例：

- 以 `load_dataset("hub_id")` 從 HF Hub 載入，確保跨機器可重現且可版控
- 以 `set_seed(42)` 與每次 shuffle 傳入 `seed` 鎖定亂數，保證結果一致
- 以 `make_balanced_split` 搭配 `datasets` API 構造平衡語料，回傳值可直接銜接 Trainer
- 以 `train_test_split(stratify_by_column="label")` 確保各 split 類別比例一致，防止 validation 失真

## 練習

1. **類別比例分析**：計算 ba_2000、ba_4000、ba_6000 三個語料的正/負比例，確認它們都是 1:1。再計算原始資料集的比例，思考為什麼訓練時需要平衡語料。

2. **學習曲線準備**：分別對 ba_2000、ba_4000、ba_6000 做 `train_test_split(test_size=0.15, stratify_by_column="label", seed=42)`，存成三個 `DatasetDict`，為後續 `06 Trainer classification_demo.ipynb` 的學習曲線實驗做準備。

3. **文字長度分析**：對 `ba_6000_split["train"]`，計算每筆 `review` 的字元數，畫出正向與負向評論的長度分布（使用 `collections.Counter` 或 pandas hist），思考是否需要截斷策略。

4. **Hub 資料集替換**：將 `DATASET_ID` 換成 `"tyqiangz/multilingual-sentiments"` 或其他中文情感資料集，觀察 `load_dataset` 的輸出格式是否需要調整 `label` 欄位名稱，練習適配不同 Hub 資料集的流程。